In [131]:
import pandas as pd
import re
import numpy as np

In [132]:
with open ('/Users/tkoti/LabWork/FlyntLab/AutoDeep/testing/Pomonella/mirdeep_Pomonella/output.mrd') as f:
    read_data = f.read().splitlines()

In [133]:
read_data

['>QFTL02000023.1_11538',
 'score total\t\t      369501.7',
 'score for star read(s)\t           3.9',
 'score for read counts\t      369494.9',
 'score for mfe\t\t           1.9',
 'score for randfold\t           1.6',
 'score for cons. seed\t          -0.6',
 'total read count\t        724760',
 'mature read count\t        718682',
 'loop read count\t\t 0',
 'star read count\t\t          6078',
 'exp                           ffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSllllllllllllMMMMMMMMMMMMMMMMMMMMMMffffffffffffffffffff',
 'obs                           ffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSllllllllllllMMMMMMMMMMMMMMMMMMMMMMffffffffffffffffffff',
 'pri_seq                       gucagaagacuguuucgccaccuggugauugccacuagcgagguauagaguuccuacguauauuggccuguaggaacuucauaccgugcucuuggguuugccaaucuucuac',
 'pri_struct                    ...((((((.((..(((((....)))))...(((..((((.(((((..((((((((((..........))))))))))..))))).))))..)))......)).))))))..\t#MM',
 'seq_5

In [134]:
entries = []
temp = []

for item in read_data:
    if item.startswith('>'):
        if temp:
            entries.append(temp)
        temp = [item]
    else:
        if not item == '':
            temp.append(item)

if temp:
    entries.append(temp)



In [135]:
entries[1]

['>QFTL02000019.1_8885',
 'score total\t\t           5.5',
 'score for star read(s)\t          -1.3',
 'score for read counts\t 0',
 'score for mfe\t\t           2.2',
 'score for randfold\t           1.6',
 'score for cons. seed\t             3',
 'miRNA with same seed\t  bmo-mir-2770',
 'total read count\t       1271611',
 'mature read count\t       1271428',
 'loop read count\t\t 0',
 'star read count\t\t           183',
 'exp                           fffffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSlllllllllllMMMMMMMMMMMMMMMMMMMMffffffffffffffffff',
 'obs                           fffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSlllllllllllMMMMMMMMMMMMMMMMMMMMffffffffffffffffff',
 'pri_seq                       cguguucaacaucgguggagagacuggcauccccuguccaucauuuggcuugacugcugcuguccacgcaucagucuugucgaaugguggguggguggugccuuguac',
 'pri_struct                    .((.(((..((....))..))))).((((((.((..(((((((((((((..(((((.(((.......))).)))))..)))))))))))))..)).))))))......\t#

In [136]:
alignments = [[item.strip() for item in entry if (item.startswith('seq_') or item.startswith('obs') or item.startswith('exp'))] for entry in entries]

In [137]:
alignments[1]

['exp                           fffffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSlllllllllllMMMMMMMMMMMMMMMMMMMMffffffffffffffffff',
 'obs                           fffffffffffffffffffffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSlllllllllllMMMMMMMMMMMMMMMMMMMMffffffffffffffffff',
 'seq_57094601_x1               ...guucaacaucgguggagagacuggcauccc...........................................................................\t0',
 'seq_48489465_x9               .....ucaacaucgguggagagacuggcauccccugu.......................................................................\t0',
 'seq_52616015_x4               ......caacaucgguggagagacuggcauccccugu.......................................................................\t0',
 'seq_57225726_x1               .......aacaucgguggagagacuggcaucc............................................................................\t0',
 'seq_54802172_x2               .........caucgguggagagacuggcauccc.................................................................

In [ ]:
# cell 7 – rewritten
import re
import pandas as pd

def parse_seq_entry(seq_line):
    s = seq_line.strip()
    # drop trailing tab/flag like "\t0" if present
    s = re.sub(r'\t.*$', '', s).strip()
    # remove surrounding quotes if any
    s = s.strip('\'"')
    m = re.match(r'^(seq_[0-9]+)_x(\d+)\s+(.*)$', s)
    consensus = re.match(r'(exp|obs)\s(.+)$', s)

    if consensus:
        return 'consensus', 1, consensus.group(2).strip()
    if m:
        return m.group(1), int(m.group(2)), m.group(3).strip()
    # fallback: first token contains id_count, rest is alignment
    parts = s.split(None, 1)
    if parts:
        m2 = re.match(r'^(seq_[0-9]+)_x(\d+)$', parts[0])
        if m2:
            alignment = parts[1].strip() if len(parts) > 1 else ''
            return m2.group(1), int(m2.group(2)), alignment
    return None, None, s

# build dataframes, dropping the 'exp' consensus only when an 'obs' is present
dfs = []
for idx, entry in enumerate(alignments):
    has_obs = any(line.strip().startswith('obs') for line in entry)
    rows = []
    for line in entry:
        if has_obs and line.strip().startswith('exp'):
            # skip the exp consensus if an obs consensus exists in this entry for more accurate downstream analysis
            continue
        name, count, aln = parse_seq_entry(line)
        if name is not None:
            rows.append((name, count, aln))

    df = pd.DataFrame(rows, columns=['sequence_name', 'sequence_count', 'alignment'])
    dfs.append(df)

# example
dfs[1].head()

,sequence_name,sequence_count,alignment
0,consensus,1,fffffffffffffffffffffffffffffffffffffSSSSSSSSS...
1,seq_57094601,1,...guucaacaucgguggagagacuggcauccc................
2,seq_48489465,9,.....ucaacaucgguggagagacuggcauccccugu............
3,seq_52616015,4,......caacaucgguggagagacuggcauccccugu............
4,seq_57225726,1,.......aacaucgguggagagacuggcaucc.................


In [109]:
combined = pd.concat(
    [df.assign(entry_index=i) for i, df in enumerate(dfs)],
    ignore_index=True
)
combined.head()

,sequence_name,sequence_count,alignment,entry_index
0,consensus,1,ffffffffffffffffffffffffffffffffffffSSSSSSSSSS...,0
1,seq_57026257,1,..................caccuggugauugccacu.............,0
2,seq_55564128,2,..........................gauugccacuagcgagguau...,0
3,seq_53349098,3,.................................Ucuagcgagguau...,0
4,seq_57407504,1,...................................Cagcgagguau...,0


In [110]:
test_case = combined[combined['entry_index'] == 2]
test_case

,sequence_name,sequence_count,alignment,entry_index
577,consensus,1,ffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSSlllll...,2
578,seq_58643452,1,.........uauuugaacuguagugccgc....................,2
579,seq_50138007,6,...........uuugaacuguagugccgcuccg................,2
580,seq_49764620,7,..........................cgcuccgaugauuucaccag...,2
581,seq_51972427,4,.................................................,2
...,...,...,...,...
977,seq_55345684,2,.................................................,2
978,seq_55503424,2,.................................................,2
979,seq_53706002,3,.................................................,2
980,seq_58007736,1,.................................................,2


Extract 5' start coordinate: For each read alignment string, find the index of the first nucleotide character (A/C/G/T/U/N, case-insensitive). Use the read's _x<count> as weight. Optionally convert index → genomic/pre-miRNA coordinate if you have a reference offset.

Dominant-start proportion:

Compute counts p_i for each 5' start position i (weighted by _x<count>).
Metric: dominant_prop = max(p_i) / sum(p_i).
Interpretation: high (e.g., >0.7–0.8) → strong 5' homogeneity.
Normalized Shannon entropy:

H = -sum_i p_i * log(p_i). Normalized H' = H / log(k) where k = number of observed start positions.
Low H' (close to 0) → homogeneous; high (close to 1) → heterogeneous. Thresholds: H' < 0.25 strong, 0.25–0.5 moderate.
Gini coefficient / inequality:

Compute Gini on the start counts distribution. Higher Gini → more concentrated (homogeneous). Use as alternative to entropy.
n90 (compactness):

Compute the minimum number of distinct start positions that together account for 90% of reads. n90 ≤ 2 → homogeneous.
Tolerance-window aggregation (handles micro-heterogeneity ±1 nt):

Merge counts across windows of ±1 (or ±2) nt and recompute dominant_prop / H'. Useful because enzymatic processing can produce 1-nt shifts.
Peak detection / KDE:

Smooth start-position counts with a small kernel, identify peak(s), compute peak height vs background (peak / median or peak / mean). High peak/background indicates homogeneity.
Statistical test vs null:

Null: uniform or background distribution from reads mapped elsewhere. Use chi-square or permutation (shuffle read starts among positions) to obtain p-value that observed concentration is non-random.
Sequence-consistency at 5' nucleotide:

For reads starting at the dominant position, compute fraction with identical first nucleotide and identical first 2–3 nt. True miRNAs often show consistent 5' bases.
Composite homogeneity score:

Combine metrics, e.g.: score = 0.5dominant_prop + 0.3(1 - H') + 0.2*(1 - n90/10) (normalize appropriately). Tune weights on labeled data.
Extraction pseudocode (5' pos + counts):

For each line in alignments:
parse count from name _x\d+
extract aln = alignment string (strip trailing \t flags)
start_idx = index of first nucleotide character in aln (0-based)
increment position counter: counts[start_idx] += count
Suggested thresholds (starting points):

dominant_prop ≥ 0.7 and normalized entropy ≤ 0.3 and n90 ≤ 2 → call highly_homogeneous.
dominant_prop 0.5–0.7 → moderately_homogeneous.
else → heterogeneous.
Notes / caveats:

Decide coordinate frame (alignment index vs genome coordinate). If alignments include flanking dots, using first-nucleotide index is sufficient for comparing within an entry.
Use _x<count> weights to avoid treating each distinct sequence string equally when counts vary widely.
Allow ±1 nt tolerance for micro-heterogeneity — recompute metrics with and without tolerance to report both strict and tolerant homogeneity.
Next step: implement one or more of these metrics in the notebook and add summary columns (dominant_prop, entropy, n90, composite_score) to your combined DataFrame. Want me to implement the strict + ±1-nt tolerant versions as notebook cells and show the results?

GPT-5 mini • 1x

In [111]:
test_alignments, test_counts = test_case['alignment'].to_numpy(), test_case['sequence_count'].to_numpy()
test_alignments



array(['ffffffffffffffffffSSSSSSSSSSSSSSSSSSSSSSSlllllllllllllllllllllllllllllMMMMMMMMMMMMMMMMMMMMMMMffffffffffffffff',
       '.........uauuugaacuguagugccgc................................................................................',
       '...........uuugaacuguagugccgcuccg............................................................................',
       '..........................cgcuccgaugauuucaccagaaga...........................................................',
       '..................................................................gggaaggcaagaugucggcauagc...................',
       '...................................................................ggaaggcaagaugucggcauagc...................',
       '....................................................................gaaggcaagaugucggcaua.....................',
       '....................................................................gaaggcaagaugucggcauag....................',
       '................................

In [142]:
# Extract 5' starting nucleotide (first nucleotide character) and its index for each alignment
five_prime_data = []

for i, aln in enumerate(test_alignments[1:]):  # skip consensus
    # Find index of first nucleotide (A, C, G, T, U, N - case insensitive)
    nucleotide_pattern = r'[ACGTUNacgtun]'
    match = re.search(nucleotide_pattern, aln)
    
    if match:
        idx = match.start()
        nucleotide = match.group().upper()
    else:
        # If no nucleotide found, record -1 and 'N'
        idx = -1
        nucleotide = 'N'
    
    # Get count from test_counts
    count = test_counts[i]
    location = test_alignments[0][idx] if idx != -1 and idx < len(test_alignments[0]) else 'N/A'

    five_prime_data.append({
        'five_prime_index': idx,
        'five_prime_nucleotide': nucleotide,
        'sequence_count': count,
        'alignment_location' : location
    })

# Create DataFrame
five_prime_df = pd.DataFrame(five_prime_data)
five_prime_df

,five_prime_index,five_prime_nucleotide,sequence_count,alignment_location
0,9,U,1,f
1,11,U,1,f
2,26,C,6,S
3,66,G,7,l
4,67,G,4,l
...,...,...,...,...
399,72,G,3,M
400,73,C,2,M
401,74,A,2,M
402,74,A,3,M


In [147]:
# from scipy import stats
# import matplotlib.pyplot as plt

# # Weight five_prime_indices by sequence_counts
# weighted_indices = five_prime_df.loc[five_prime_df.index.repeat(five_prime_df['sequence_count']),'five_prime_index'].values

# # Create KDE plot with weighted indices
# kde = stats.gaussian_kde(weighted_indices)
# x_range = np.linspace(weighted_indices.min(), weighted_indices.max(), 200)

# plt.figure(figsize=(12, 5))
# plt.plot(x_range, kde(x_range), linewidth=2)
# plt.fill_between(x_range, kde(x_range), alpha=0.3)
# plt.xlabel('5\' Start Index')
# plt.ylabel('Density')
# plt.title('KDE Plot of 5\' Start Positions (Weighted by Sequence Count)')
# plt.grid(True, alpha=0.3)
# plt.show()

In [148]:
five_prime_df['sequence_count'].describe()

count       404.000000
mean       1071.029703
std       12178.139122
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max      217177.000000
Name: sequence_count, dtype: float64

In [181]:
five_prime_df.groupby('five_prime_index')['sequence_count'].sum().sort_values(ascending=False)

five_prime_index
70    427429
69      4133
71       863
72       210
68        34
66         7
26         6
74         5
67         4
73         2
9          1
11         1
75         1
Name: sequence_count, dtype: int64

In [192]:
def is_homogeneous(five_prime_index):
    """
    Improved homogeneity metrics for a given 5' start position.
    
    Returns:
    - max_prop: Proportion of reads at single most abundant position
    - top1/2/3_prop: Cumulative proportion in top N positions
    - normalized_entropy: Shannon entropy normalized by max entropy
    - gini_coefficient: Inequality measure (0=uniform, 1=concentrated)
    - n_distinct_positions: Number of different 5' start positions
    - coefficient_variation: Relative spread of counts
    - total_count: Total reads in window
    """
    # Determine the mature miRNA size range (typically 20-24 nt)
    alignment_position = five_prime_df.loc[five_prime_df['five_prime_index'] == five_prime_index, 'alignment_location'].iloc[0]
    if alignment_position == 'M' or alignment_position == 'S':
        miRNA_size = test_alignments[0].count(alignment_position)
    else:
        miRNA_size = 22  # default to 22nt if alignment location is not M or S
    
    
    min_idx = five_prime_index - miRNA_size
    max_idx = five_prime_index + miRNA_size
    
    # Filter reads within the tolerance window
    filtered = five_prime_df[
        (five_prime_df['five_prime_index'] >= min_idx) & 
        (five_prime_df['five_prime_index'] <= max_idx)
    ]
    if filtered.empty:
        return None
    
    counts = filtered['sequence_count'].values
    total = counts.sum()
    proportions = counts / total
    
    # Metric 1: Maximum proportion (reads at most abundant position)
    max_prop = proportions.max()
    # print entry or entries that achieve this max_prop
    max_positions = np.where(proportions == max_prop)[0]
    print("\nMAX_PROP entries in filtered window:")
    print(filtered.iloc[max_positions])
    
    # determine dominant nucleotide among filtered
    max_idx_in_filtered = np.argmax(counts)
    dominant_nucleotide = filtered.iloc[max_idx_in_filtered]['five_prime_nucleotide']
    
    # restrict to rows with dominant nucleotide
    nucleotide_filtered = filtered[filtered['five_prime_nucleotide'] == dominant_nucleotide]
    if nucleotide_filtered.empty:
        nucleotide_filtered = filtered
    
    # further restrict to exact five_prime_index for top-N metrics
    pos_filtered = nucleotide_filtered[nucleotide_filtered['five_prime_index'] == five_prime_index]
    use_filtered = pos_filtered if not pos_filtered.empty else nucleotide_filtered
    if pos_filtered.empty:
        print(f"WARNING: no reads exactly at position {five_prime_index}; using {len(use_filtered)} reads for top-N metrics")
    
    # compute proportions over the chosen subset
    sub_counts = use_filtered['sequence_count'].values
    sub_total = sub_counts.sum()
    sub_props = sub_counts / sub_total if sub_total > 0 else sub_counts
    
    # sort by these proportions
    sorted_idx = np.argsort(sub_props)[::-1]
    sorted_props = sub_props[sorted_idx]
    
    print(f"\n{'='*80}")
    print(f"5' Index: {five_prime_index} | Dominant Nucleotide: {dominant_nucleotide}")
    print(f"Rows at exact position: {len(pos_filtered)}; using {len(use_filtered)} rows for top-N")
    print(f"{'='*80}\n")
    
    # TOP 1
    top1_entries = use_filtered.iloc[sorted_idx[:1]]
    print("TOP1 entries:")
    print(top1_entries)
    top1_prop = sorted_props[0] if len(sorted_props) > 0 else 0
    print(f"TOP1 proportion: {top1_prop:.4f}\n")
    
    # TOP 2
    count_n = min(2, len(sorted_idx))
    top2_entries = use_filtered.iloc[sorted_idx[:count_n]]
    print("TOP2 entries:")
    print(top2_entries)
    top2_prop = sorted_props[:count_n].sum() if len(sorted_props) >= count_n else sorted_props.sum()
    print(f"TOP2 proportion: {top2_prop:.4f}\n")
    
    # TOP 3
    count_n = min(3, len(sorted_idx))
    top3_entries = use_filtered.iloc[sorted_idx[:count_n]]
    print("TOP3 entries:")
    print(top3_entries)
    top3_prop = sorted_props[:count_n].sum() if len(sorted_props) >= count_n else sorted_props.sum()
    print(f"TOP3 proportion: {top3_prop:.4f}\n")
    
    # Metric 3: Normalized Shannon entropy (still over full window)
    entropy = -np.sum(proportions * np.log2(proportions + 1e-10))
    max_entropy = np.log2(len(proportions))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    
    # Metric 4: Gini coefficient (0=uniform, 1=concentrated)
    sorted_counts = np.sort(counts)
    n = len(sorted_counts)
    gini = (2 * np.sum(np.arange(1, n+1) * sorted_counts)) / (n * sorted_counts.sum()) - (n + 1) / n
    
    # Metric 5: Number of distinct positions
    n_distinct_positions = len(filtered)
    
    # Metric 6: Coefficient of variation of counts
    cv = np.std(counts) / np.mean(counts) if np.mean(counts) > 0 else 0
    
    return {
        'star_or_mature' : alignment_position,
        'five_prime_index': five_prime_index,
        'max_prop': max_prop,
        'top1_prop': top1_prop,
        'top2_prop': top2_prop,
        'top3_prop': top3_prop,
        'normalized_entropy': normalized_entropy,
        'gini_coefficient': gini,
        'n_distinct_positions': n_distinct_positions,
        'coefficient_variation': cv,
        'total_count': total
    }

In [193]:
is_homogeneous(70)


MAX_PROP entries in filtered window:
     five_prime_index five_prime_nucleotide  sequence_count alignment_location
266                70                     U          217177                  M

5' Index: 70 | Dominant Nucleotide: U
Rows at exact position: 6; using 6 rows for top-N

TOP1 entries:
     five_prime_index five_prime_nucleotide  sequence_count alignment_location
266                70                     U          217177                  M
TOP1 proportion: 0.9841

TOP2 entries:
     five_prime_index five_prime_nucleotide  sequence_count alignment_location
266                70                     U          217177                  M
344                70                     U            3481                  M
TOP2 proportion: 0.9999

TOP3 entries:
     five_prime_index five_prime_nucleotide  sequence_count alignment_location
266                70                     U          217177                  M
344                70                     U            3481          

{'star_or_mature': 'M',
 'five_prime_index': 70,
 'max_prop': np.float64(0.5019251747217395),
 'top1_prop': np.float64(0.9841441751708387),
 'top2_prop': np.float64(0.999918432453008),
 'top3_prop': np.float64(0.9999546846961156),
 'normalized_entropy': np.float64(0.2603068730666469),
 'gini_coefficient': np.float64(0.9877970965792633),
 'n_distinct_positions': 401,
 'coefficient_variation': np.float64(11.314054213195941),
 'total_count': np.int64(432688)}